In [1]:
# conda activate anndata

import os
import sys
import anndata as ad

sys.path.append("/mnt/lareaulab/reliscu/code")

from junction2psi import *

In [2]:
adata = ad.read_h5ad("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_SJ_counts.h5ad")

In [3]:
adata.shape

(772, 233295)

In [4]:
SJ_counts_table = pd.DataFrame(adata.X.T, columns=adata.obs_names, index=adata.var_names)

In [5]:
SJ_counts_table.shape

(233295, 772)

In [7]:
events_i1 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I1' in x])
events_i2 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I2' in x])
events_se = pd.Index([x[:-3] for x in SJ_counts_table.index if '_SE' in x])

events = events_i1.intersection(events_i2).intersection(events_se)
i1_events = [x + '_I1' for x in events]
I1_table = SJ_counts_table.loc[i1_events]
I1_table.index = events

i2_events = [x + '_I2' for x in events]
I2_table = SJ_counts_table.loc[i2_events]
I2_table.index = events

se_events = [x + '_SE' for x in events]
SE_table = SJ_counts_table.loc[se_events]
SE_table.index = events

I1_filt = I1_table.index[I1_table.sum(axis=1) > 0]
I2_filt = I2_table.index[I2_table.sum(axis=1) > 0]
SE_filt = SE_table.index[SE_table.sum(axis=1) > 0]
filtered_events = I1_filt.intersection(I2_filt).intersection(SE_filt)

I1_table = I1_table.loc[filtered_events]
I2_table = I2_table.loc[filtered_events]
SE_table = SE_table.loc[filtered_events]

psi = ((I1_table + I2_table) /(2*SE_table + I1_table + I2_table)).fillna(0)
reads = SE_table + I1_table + I2_table

In [8]:
psi.shape[0]

47058

In [9]:
psi.head()

,GTEX-1117F-0011-R10b-SM-GI4VE,GTEX-1117F-0011-R3a-SM-GJ3PJ,GTEX-1117F-3226-SM-5N9CT,GTEX-111FC-0011-R10a-SM-GIN8G,GTEX-111FC-0011-R3b-SM-GJ3PN,GTEX-111FC-3126-SM-5GZZ2,GTEX-1128S-2726-SM-5H12C,GTEX-117XS-0011-R10b-SM-GIN8Z,GTEX-117XS-0011-R3a-SM-GIN8W,GTEX-117XS-3026-SM-5N9CA,...,GTEX-ZXG5-0011-R10a-SM-57WDD,GTEX-ZYFD-0011-R10a-SM-GPI91,GTEX-ZYFD-0011-R3b-SM-GPRX3,GTEX-ZYFD-3026-SM-5E44C,GTEX-ZYY3-0011-R10a-SM-GNTAZ,GTEX-ZYY3-0011-R3a-SM-GQ1CX,GTEX-ZYY3-3126-SM-5MR6M,GTEX-ZZPT-0011-R10b-SM-GPI8B,GTEX-ZZPT-0011-R3a-SM-GOQYT,GTEX-ZZPT-3026-SM-5GZXH
ENSG00000292994_other_1,1.0,0.000000,1.000000,1.000000,1.000000,0.818182,1.000000,0.666667,1.0,0.333333,...,1.0,1.000000,0.0,0.666667,1.000000,1.000000,0.0,1.0,0.428571,0.200000
ENSG00000290385_other_1,0.0,0.090909,0.500000,0.272727,0.111111,0.500000,0.043478,0.333333,0.5,0.200000,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.333333
ENSG00000290385_other_2,0.0,0.090909,0.333333,0.272727,0.272727,0.333333,0.043478,0.200000,0.5,0.272727,...,0.0,0.142857,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.333333
ENSG00000290385_other_3,1.0,0.750000,1.000000,1.000000,1.000000,1.000000,1.000000,0.428571,1.0,1.000000,...,1.0,1.000000,1.0,1.000000,1.000000,0.714286,1.0,1.0,0.428571,1.000000
ENSG00000290385_other_4,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,...,1.0,1.000000,1.0,1.000000,0.846154,1.000000,1.0,0.0,1.000000,1.000000


In [10]:
psi.to_csv(f"data/GTEx_cortex_exon_PSI.csv")
reads.to_csv(f"data/GTEx_cortex_exon_counts.csv")